In [1]:
import os
import re
from dotenv import load_dotenv
load_dotenv()  # picks up .env from the repo root
# Nemotron reasons out loud by default; these demos want direct answers
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}
from openai import OpenAI

API_KEY = os.environ["NVIDIA_API_KEY"]
# Replace with your actual NIM key
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=API_KEY
)
MODEL = "nvidia/nemotron-3-super-120b-a12b"

def llm_rerank(query, documents, top_k=3, verbose=True):
    """
    Scores each document's relevance to the query using the LLM.
    Returns top_k documents sorted by descending relevance score.
    """
    scored_docs = []
    for doc in documents:
        prompt = f"""On a scale of 1 to 10, how relevant is the following document to the query?
Query: {query}
Document: {doc}
Respond with only the number.
Relevance score (1-10):"""

        response = client.chat.completions.create(
            model=MODEL,
            extra_body=NO_THINK,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300,
            temperature=0
        )

        print('response ', response)

        raw_score = response.choices[0].message.content.strip()
        # Models often wrap the number in prose ("a score of 9 out of 10"), so take the first number
        match = re.search(r"\d+(?:\.\d+)?", raw_score)
        try:
            score = float(match.group() if match else raw_score)
        except ValueError:
            # If the model doesn't return a number, assign 0 and optionally log
            print(f"Warning: Could not parse score for doc: '{doc[:50]}...' -> '{raw_score}'")
            score = 0.0

        scored_docs.append((doc, score))
        if verbose:
            print(f"Score {score:4.1f} | {doc[:60]}...")

    # Sort by score descending
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, _ in scored_docs[:top_k]]
    return top_docs

# ========== Test Data ==========
query = "What are the health benefits of regular exercise?"

documents = [
    "Regular physical activity can reduce the risk of heart disease and stroke.",
    "Exercise helps control weight, improves mental health, and strengthens bones.",
    "The history of the Olympic Games dates back to ancient Greece.",
    "A balanced diet includes fruits, vegetables, and whole grains.",
    "Cardiovascular exercises like running and swimming increase heart rate and lung capacity.",
    "The stock market can be volatile; diversification is key.",
    "Strength training builds muscle mass and boosts metabolism.",
    "Regular exercise is linked to better sleep and reduced anxiety.",
]

print("=== Testing LLM Reranker ===")
top_3 = llm_rerank(query, documents, top_k=3, verbose=True)

print("\n=== Top 3 Documents ===")
for i, doc in enumerate(top_3, 1):
    print(f"{i}. {doc}")

=== Testing LLM Reranker ===


response  ChatCompletion(id='chatcmpl-46e7beb2-248d-45f3-9c8e-66383bb13a4d', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='9', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=None))], created=1790241924, model='nvidia/nemotron-3-super-120b-a12b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=80, total_tokens=82, completion_tokens_details=None, prompt_tokens_details=None))
Score  9.0 | Regular physical activity can reduce the risk of heart disea...


response  ChatCompletion(id='chatcmpl-349a35b6-5755-4b34-a346-4b82e9326ee7', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='10', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=None))], created=1790241925, model='nvidia/nemotron-3-super-120b-a12b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=3, prompt_tokens=81, total_tokens=84, completion_tokens_details=None, prompt_tokens_details=None))
Score 10.0 | Exercise helps control weight, improves mental health, and s...


response  ChatCompletion(id='chatcmpl-5c00570d-e8d8-41e4-a038-6a4f71aeaf5e', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='1', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=None))], created=1790241925, model='nvidia/nemotron-3-super-120b-a12b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=79, total_tokens=81, completion_tokens_details=None, prompt_tokens_details=None))
Score  1.0 | The history of the Olympic Games dates back to ancient Greec...


response  ChatCompletion(id='chatcmpl-940f27c0-bdd6-4708-b74d-d82ca6a2e1fb', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='1', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=None))], created=1790241926, model='nvidia/nemotron-3-super-120b-a12b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=79, total_tokens=81, completion_tokens_details=None, prompt_tokens_details=None))
Score  1.0 | A balanced diet includes fruits, vegetables, and whole grain...


response  ChatCompletion(id='chatcmpl-2a3dc0e6-a4b3-4c40-b061-088d4695d69b', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='8', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=None))], created=1790241926, model='nvidia/nemotron-3-super-120b-a12b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=81, total_tokens=83, completion_tokens_details=None, prompt_tokens_details=None))
Score  8.0 | Cardiovascular exercises like running and swimming increase ...


response  ChatCompletion(id='chatcmpl-a2f0ed85-07ca-4ce2-af8b-0b84c8432e52', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='1', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=None))], created=1790241927, model='nvidia/nemotron-3-super-120b-a12b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=78, total_tokens=80, completion_tokens_details=None, prompt_tokens_details=None))
Score  1.0 | The stock market can be volatile; diversification is key....


response  ChatCompletion(id='chatcmpl-78d8df60-0bb5-4196-8c10-95753d325777', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='7', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=None))], created=1790241927, model='nvidia/nemotron-3-super-120b-a12b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=77, total_tokens=79, completion_tokens_details=None, prompt_tokens_details=None))
Score  7.0 | Strength training builds muscle mass and boosts metabolism....


response  ChatCompletion(id='chatcmpl-5555e5dd-a790-4564-b932-25f88599c166', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='8', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content=None))], created=1790241928, model='nvidia/nemotron-3-super-120b-a12b', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=78, total_tokens=80, completion_tokens_details=None, prompt_tokens_details=None))
Score  8.0 | Regular exercise is linked to better sleep and reduced anxie...

=== Top 3 Documents ===
1. Exercise helps control weight, improves mental health, and strengthens bones.
2. Regular physical activity can reduce the risk of heart disease and stroke.
3. Cardiovascular exercises like running and swimming increase heart rate and lung capacity.
